### 角点检测
#### 绘制关键点

In [2]:
# 绘制关键点
import cv2
import random

In [ ]:

img = cv2.imread('../data/images/fox.jpg')
img = cv2.resize(img, (400, 600))
height, width = img.shape[:2]

# 随机生成关键点
key_points = []
num_keypoints = 50
for point in range(num_keypoints):
    x = random.randint(0, width - 1)
    y = random.randint(0, height - 1)
    # pt: 坐标位置，格式为 (x, y)
    # size: 关键点的直径（这里设置为 1）
    key_points.append(cv2.KeyPoint(x, y, size=1))

key_point_image = cv2.drawKeypoints(img, key_points, None, color=(0, 255, 0), flags=0)

cv2.imshow('Random Keypoints', key_point_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [1]:
import numpy as np

#### 使用Harris 角点检测

In [20]:
img = cv2.imread('../data/images/esma.jpg')
# resize()中的dsize以(widths heigth)的形式表示
img = cv2.resize(img, (400, 600))

# 将原始图片转为灰度图
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Harris角点检测
gray_img = np.float32(gray_img)
# dst是一个灰度图像(角点响应值)，其中每个像素的强度值代表了该点作为角点的可能性
dst = cv2.cornerHarris(gray_img, blockSize=2, ksize=3, k=0.04)
print(dst.shape)

# 先膨胀角点响应值矩阵，再用阈值提取角点
# 通过膨胀操作，使得角点区域更连续，原来可能会漏检的角点也能被检测出来
dst = cv2.dilate(dst, None)
thresh = 0.01 * dst.max()
# opencv读取的是BGR图像
# 如果当前像素的值大于thresh，则认为该点是角点，在图像上用红色画出
img[dst > thresh] = [0, 0, 255]

# 绘制圆圈标记点
# for i in range(dst.shape[0]): # H
#     for j in range(dst.shape[1]): # W
#         if dst[i, j] > thresh:
#             # opencv的坐标格式为
#             cv2.circle(img, (j, i), radius=2, color=(0, 255, 0), thickness=1)

cv2.imshow('Harris Corners', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

(600, 400)


### 特征检测


#### SIFT
高斯金字塔构建过程中，一般首先将图像扩大一倍，将原始图像扩大一倍后再滤波能够保留更多的信息便于后续特征提取与匹配。然后在扩大的图像的基础上构建高斯金字塔，然后对该尺寸下图像进行不同参数的高斯模糊（每一幅模糊的图像称为一个层（layer）），几幅模糊图像的图像集合构成了一个组（octave）。然后对该组下的倒数第三张图像进行降采样（长宽分别缩小一倍）作为下一个组的初始图像，在初始图像的基础上完成属于这个组的高斯模糊处理，依次类推完成整个算法所需要的所有组构建。

In [4]:
img = cv2.imread('../data/images/fox.jpg', 0)
img = cv2.resize(img, (400, 600))

# 创建SIFT实例对象对特征点周围区域的数学描述，用于特征匹配
sift = cv2.SIFT_create(nfeatures=100)

# 特征子：图像中的显著点，具有尺度、旋转不变性，包含位置坐标、尺度、方向和显著性
# 描述子：对特征点周围区域的数学描述，用于特征匹配，描述特征点周围的梯度分布
keypoints, descripters = sift.detectAndCompute(img, None)
print(descripters)

img_with_keypoints = cv2.drawKeypoints(img, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

cv2.imshow('SIFT', img_with_keypoints)
cv2.waitKey(0)
cv2.destroyAllWindows()

[[30. 13. 11. ... 35.  5. 24.]
 [15.  3.  1. ...  0. 32. 57.]
 [ 4. 54. 23. ...  8.  0.  0.]
 ...
 [49.  2.  1. ... 33.  3. 25.]
 [12. 27. 10. ... 31. 47. 58.]
 [44. 23. 45. ... 37. 25. 19.]]


#### ORB
- 使用FAST来生成特征子，并通过金字塔、质心标定等方法解决尺度不变和旋转不变
- 使用BRIEF用来构造描述子的。ORB在BRIEF基础上引入oFAST的旋转角度和机器学习解决了旋转特性和特征点难以区分的问题

In [6]:
img_fox = cv2.imread('../data/images/fox.jpg', 0)
img_fox = cv2.resize(img_fox, (400, 600))

(h, w) = img_fox.shape[:2]
center = (w // 2, h // 2)
M = cv2.getRotationMatrix2D(center, 90, 0.5)
img_rotated_fox = cv2.warpAffine(img_fox, M, (w, h))

# 需要创建ORB实例
orb = cv2.ORB_create(nfeatures=100)

kpFox, desFox = orb.detectAndCompute(img_fox, None)
kpRotateFox, desRotateFox = orb.detectAndCompute(img_rotated_fox, None)

# 使用BF匹配方法
bf = cv2.BFMatcher_create(cv2.NORM_HAMMING, crossCheck=True)

matches = bf.match(desFox, desRotateFox)
matchImg = cv2.drawMatches(img_fox, kpFox, img_rotated_fox, kpRotateFox, matches, None)

cv2.imshow('Fox', img_fox)
cv2.imshow('Rotate Fox', img_rotated_fox)
cv2.imshow('match', matchImg)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [9]:
# 使用FLANN匹配器

sift = cv2.SIFT_create(nfeatures=100)

kpFox, desFox = sift.detectAndCompute(img_fox, None)
kpRotateFox, desRotateFox = sift.detectAndCompute(img_rotated_fox, None)

# FLANN参数设置
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

# 创建FLANN匹配器
flann = cv2.FlannBasedMatcher(indexParams=index_params, searchParams=search_params)

# 使用KNN匹配
matches = flann.knnMatch(desFox, desRotateFox, k=2)

# 应用Lowe's比率测试来筛选好的匹配点
good_matches = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good_matches.append(m)

matchImg = cv2.drawMatches(img_fox, kpFox, img_rotated_fox, kpRotateFox, good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

cv2.imshow('match', matchImg)
cv2.waitKey(0)
cv2.destroyAllWindows()